# 🛡️ Session 6: Advanced Error Handling, Context Managers & Pathlib
### *Write bulletproof code that handles anything*

---

| Info | Detail |
|------|--------|
| ⏱️ Duration | ~60 minutes |
| 📝 Exercises | 15 (with hints & solutions) |
| 🎯 Goal | Master custom exceptions, context managers, pathlib, logging |
| 📋 Prerequisites | Sessions 1–5 |

---

## 6.1 Exception Hierarchy & Custom Exceptions 🏗️

In [ ]:
# Python's exception hierarchy (partial)
# BaseException
#  ├── KeyboardInterrupt
#  ├── SystemExit
#  └── Exception
#       ├── ValueError
#       ├── TypeError
#       ├── KeyError
#       ├── FileNotFoundError (subclass of OSError)
#       └── ... your custom exceptions go here!

# Custom exception hierarchy
class AppError(Exception):
    """Base error for our application."""
    pass

class ValidationError(AppError):
    """Input validation failed."""
    def __init__(self, field, value, message):
        self.field = field
        self.value = value
        super().__init__(f"{field}: {message} (got {value!r})")

class NotFoundError(AppError):
    """Resource not found."""
    def __init__(self, resource, id):
        self.resource = resource
        self.id = id
        super().__init__(f"{resource} #{id} not found")

# Usage
try:
    raise ValidationError("age", -5, "must be positive")
except ValidationError as e:
    print(f"❌ {e}")
    print(f"   Field: {e.field}, Value: {e.value}")

try:
    raise NotFoundError("User", 42)
except AppError as e:  # catches ALL app errors
    print(f"❌ {e}")

### ✏️ Exercise 1 — Custom Exception Hierarchy for Banking 🏦

In [ ]:
# TODO: Create a banking exception hierarchy
class BankError(Exception): pass

class InsufficientFundsError(BankError):
    def __init__(self, balance, amount):
        self.balance = balance
        self.amount = amount
        self.shortfall = amount - balance
        super().__init__(f"Need ₹{amount:,} but only ₹{balance:,} available (₹{self.shortfall:,} short)")

class InvalidAmountError(BankError):
    def __init__(self, amount):
        super().__init__(f"Invalid amount: ₹{amount}")

class AccountLockedError(BankError):
    def __init__(self, account_id):
        super().__init__(f"Account {account_id} is locked")

# Test
def withdraw(balance, amount):
    if amount <= 0:
        raise InvalidAmountError(amount)
    if amount > balance:
        raise InsufficientFundsError(balance, amount)
    return balance - amount

for amt in [-100, 5000, 500]:
    try:
        result = withdraw(1000, amt)
        print(f"  ✅ Withdrew ₹{amt:,} → ₹{result:,}")
    except BankError as e:
        print(f"  ❌ {type(e).__name__}: {e}")

---
## 6.2 Advanced try/except Patterns 🎯

In [ ]:
# else clause — runs ONLY if no exception
# finally clause — runs ALWAYS

def divide(a, b):
    try:
        result = a / b
    except ZeroDivisionError:
        print("  ❌ Division by zero!")
        return None
    except TypeError as e:
        print(f"  ❌ Type error: {e}")
        return None
    else:
        print(f"  ✅ Success: {result}")  # only if NO exception
        return result
    finally:
        print("  🔚 Cleanup done")  # ALWAYS runs

divide(10, 3)
divide(10, 0)
divide(10, "a")

In [ ]:
# Exception chaining — 'from' keyword
def process_data(data):
    try:
        return int(data)
    except ValueError as original:
        raise ValidationError("data", data, "must be a number") from original

try:
    process_data("abc")
except ValidationError as e:
    print(f"❌ {e}")
    print(f"   Caused by: {e.__cause__}")

### ✏️ Exercise 2 — `try/except/else/finally` Practice 🎯

In [ ]:
# TODO: Predict the output for each call

def safe_parse(text):
    print(f"  Parsing '{text}'...")
    try:
        value = int(text)
    except ValueError:
        print("  ❌ Not a number")
        return None
    else:
        print(f"  ✅ Got {value}")
        return value
    finally:
        print("  🔚 Done")

print(safe_parse("42"))    # What prints? What returns?
print("---")
print(safe_parse("abc"))   # What prints? What returns?

### ✏️ Exercise 3 — Multi-Exception Handling 🎪

In [ ]:
def safe_get(data, key, index=None):
    """Safely get nested data with detailed error handling."""
    try:
        value = data[key]
        if index is not None:
            value = value[index]
        return value
    except KeyError:
        return f"❌ Key '{key}' not found"
    except (IndexError, TypeError) as e:
        return f"❌ Index error on '{key}': {e}"

data = {
    "users": ["Anita", "Rahul", "Sneha"],
    "count": 3,
}

print(safe_get(data, "users", 0))     # Anita
print(safe_get(data, "users", 10))    # IndexError
print(safe_get(data, "missing"))      # KeyError
print(safe_get(data, "count", 0))     # TypeError (int not subscriptable)

---
## 6.3 Context Managers — `with` Statement 🚪

Context managers handle **setup** and **cleanup** automatically.

In [ ]:
# Class-based context manager
class FileManager:
    def __init__(self, filename, mode="r"):
        self.filename = filename
        self.mode = mode
        self.file = None
    
    def __enter__(self):
        print(f"  📂 Opening {self.filename}")
        self.file = open(self.filename, self.mode)
        return self.file
    
    def __exit__(self, exc_type, exc_val, exc_tb):
        print(f"  📁 Closing {self.filename}")
        if self.file:
            self.file.close()
        if exc_type:
            print(f"  ⚠️ Exception: {exc_val}")
        return False  # don't suppress exceptions

# Test with a temp file
import tempfile, os
tmp = os.path.join(tempfile.gettempdir(), "test_ctx.txt")

with FileManager(tmp, "w") as f:
    f.write("Hello, Context Manager!")

with FileManager(tmp, "r") as f:
    print(f"  Content: {f.read()}")

In [ ]:
# Generator-based context manager (simpler!)
from contextlib import contextmanager
import time

@contextmanager
def timer(label="Timer"):
    start = time.time()
    print(f"  ⏱️ {label} started")
    try:
        yield  # body of 'with' block runs here
    finally:
        elapsed = time.time() - start
        print(f"  ⏱️ {label}: {elapsed:.4f}s")

with timer("Calculation"):
    total = sum(range(1_000_000))

### ✏️ Exercise 4 — Write a `@contextmanager` for Temp Directory 📁

In [ ]:
from contextlib import contextmanager
import tempfile, shutil, os

@contextmanager
def temp_directory(prefix="tmp_"):
    """Create a temp directory, use it, then auto-clean up."""
    tmpdir = tempfile.mkdtemp(prefix=prefix)
    print(f"  📂 Created: {tmpdir}")
    try:
        yield tmpdir
    finally:
        shutil.rmtree(tmpdir)
        print(f"  🗑️ Cleaned up: {tmpdir}")

with temp_directory("myapp_") as d:
    filepath = os.path.join(d, "data.txt")
    with open(filepath, "w") as f:
        f.write("temporary data")
    print(f"  📄 File exists: {os.path.exists(filepath)}")

# After with block, directory is gone!

### ✏️ Exercise 5 — Database Connection Simulator 🗄️

In [ ]:
class DatabaseConnection:
    """Simulated database connection with context manager."""
    def __init__(self, db_name):
        self.db_name = db_name
        self.connected = False
        self._data = {}
    
    def __enter__(self):
        print(f"  🔌 Connecting to '{self.db_name}'...")
        self.connected = True
        return self
    
    def __exit__(self, exc_type, exc_val, exc_tb):
        if exc_type:
            print(f"  ⚠️ Rolling back due to: {exc_val}")
        else:
            print(f"  💾 Committing changes...")
        self.connected = False
        print(f"  🔌 Disconnected from '{self.db_name}'")
        return False
    
    def execute(self, query):
        if not self.connected:
            raise RuntimeError("Not connected!")
        print(f"  📝 Executing: {query}")
        return {"status": "ok", "query": query}

# Normal usage
with DatabaseConnection("users_db") as db:
    db.execute("SELECT * FROM users")
    db.execute("INSERT INTO users VALUES ('Priya', 28)")

print()

# With error — auto-rollback!
try:
    with DatabaseConnection("orders_db") as db:
        db.execute("DELETE FROM orders")
        raise ValueError("Oops!")
except ValueError:
    pass

---
## 6.4 `pathlib` — Modern File Paths 📂

In [ ]:
from pathlib import Path

# Creating paths
p = Path.home() / "Documents" / "project"
print(f"Path: {p}")
print(f"Parts: {p.parts}")
print(f"Parent: {p.parent}")
print(f"Name: {p.name}")

# File operations
f = Path("report.2024.data.csv")
print(f"\nStem: {f.stem}")         # report.2024.data
print(f"Suffix: {f.suffix}")       # .csv
print(f"Suffixes: {f.suffixes}")   # ['.2024', '.data', '.csv']

# Change extension
print(f"As JSON: {f.with_suffix('.json')}")

# Useful checks
cwd = Path.cwd()
print(f"\nCWD: {cwd}")
print(f"Exists: {cwd.exists()}")
print(f"Is dir: {cwd.is_dir()}")

In [ ]:
from pathlib import Path
import tempfile

# Read/Write with pathlib (no need for open()!)
tmp = Path(tempfile.gettempdir()) / "pathlib_test.txt"

tmp.write_text("Hello from pathlib! 🐍\nSecond line.")
print(tmp.read_text())

# Glob — find files
cwd = Path.cwd()
print(f"\n📄 Python files in parent:")
for f in cwd.parent.glob("*.ipynb"):
    print(f"  {f.name} ({f.stat().st_size:,} bytes)")

### ✏️ Exercise 6 — `pathlib` File Explorer 📂

In [ ]:
from pathlib import Path

def explore(directory, pattern="*", max_depth=2):
    """Display directory tree with file sizes."""
    root = Path(directory)
    if not root.exists():
        print(f"❌ {directory} not found")
        return
    
    print(f"📂 {root}")
    total_size = 0
    file_count = 0
    
    for item in sorted(root.rglob(pattern)):
        # Skip if too deep
        depth = len(item.relative_to(root).parts)
        if depth > max_depth:
            continue
        
        indent = "  " * depth
        if item.is_file():
            size = item.stat().st_size
            total_size += size
            file_count += 1
            print(f"{indent}📄 {item.name} ({size:,} bytes)")
        elif item.is_dir():
            print(f"{indent}📁 {item.name}/")
    
    print(f"\n📊 {file_count} files, {total_size:,} bytes total")

explore(Path.cwd().parent, "*.ipynb", max_depth=2)

### ✏️ Exercise 7 — File Backup with `pathlib` 💾

In [ ]:
from pathlib import Path
from datetime import datetime
import shutil, tempfile

def backup_file(filepath):
    """Create a timestamped backup of a file."""
    src = Path(filepath)
    if not src.exists():
        raise FileNotFoundError(f"{filepath} not found")
    
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    backup_name = f"{src.stem}_{timestamp}{src.suffix}"
    backup_dir = src.parent / "backups"
    backup_dir.mkdir(exist_ok=True)
    
    dest = backup_dir / backup_name
    shutil.copy2(src, dest)
    print(f"  💾 Backed up: {src.name} → backups/{backup_name}")
    print(f"     Size: {dest.stat().st_size:,} bytes")
    return dest

# Demo with temp file
tmp = Path(tempfile.gettempdir()) / "important_data.txt"
tmp.write_text("Critical business data\n" * 100)
backup_file(tmp)

---
## 6.5 `logging` Module — Professional Output 📋

In [ ]:
import logging

# Configure logging
logging.basicConfig(
    level=logging.DEBUG,
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
    datefmt="%H:%M:%S"
)

logger = logging.getLogger("MyApp")

# Log levels (from least to most severe)
logger.debug("Detailed debugging info")       # 10
logger.info("General information")             # 20
logger.warning("Something unexpected")         # 30
logger.error("Something failed")               # 40
logger.critical("System is down!")             # 50

In [ ]:
import logging

# Logging exceptions with traceback
logger = logging.getLogger("ErrorDemo")

def risky_operation(x):
    try:
        return 100 / x
    except ZeroDivisionError:
        logger.exception("Division by zero!")  # includes traceback!
        return None

risky_operation(0)

### ✏️ Exercise 8 — Logging to File 📝

In [ ]:
import logging
from pathlib import Path
import tempfile

# Log to both console and file
log_file = Path(tempfile.gettempdir()) / "app.log"

logger = logging.getLogger("FileLogger")
logger.setLevel(logging.DEBUG)
logger.handlers.clear()  # clear any existing handlers

# Console handler (INFO and above)
console = logging.StreamHandler()
console.setLevel(logging.INFO)
console.setFormatter(logging.Formatter("%(levelname)s: %(message)s"))

# File handler (DEBUG and above)
file_handler = logging.FileHandler(log_file)
file_handler.setLevel(logging.DEBUG)
file_handler.setFormatter(logging.Formatter("%(asctime)s [%(levelname)s] %(message)s"))

logger.addHandler(console)
logger.addHandler(file_handler)

# Test
logger.debug("This goes to file only")
logger.info("This goes to both")
logger.warning("Warning: something unusual")
logger.error("Error occurred!")

print(f"\n📄 Log file content:")
print(log_file.read_text())

### ✏️ Exercise 9 — Suppress Context Manager 🤫

In [ ]:
from contextlib import suppress

# suppress — ignore specific exceptions
# Instead of:
try:
    result = int("abc")
except ValueError:
    pass

# Use:
with suppress(ValueError):
    result = int("abc")  # silently ignored

# Practical: safely delete files
from pathlib import Path
with suppress(FileNotFoundError):
    Path("/nonexistent/file.txt").unlink()

# Safely access nested dicts
data = {"users": [{"name": "Anita"}]}
with suppress(KeyError, IndexError, TypeError):
    email = data["users"][0]["email"]

print("✅ No crashes!")

### ✏️ Exercise 10 — `ExitStack` for Multiple Resources 📚

In [ ]:
from contextlib import ExitStack
from pathlib import Path
import tempfile

# ExitStack — manage variable number of context managers
tmpdir = Path(tempfile.gettempdir())
files_to_create = ["data1.txt", "data2.txt", "data3.txt"]

with ExitStack() as stack:
    # Open multiple files dynamically
    file_handles = []
    for name in files_to_create:
        path = tmpdir / name
        f = stack.enter_context(open(path, "w"))
        file_handles.append(f)
    
    # Write to all files
    for i, f in enumerate(file_handles):
        f.write(f"Data for file {i+1}")
    
    print(f"  📝 Wrote to {len(file_handles)} files")

# All files are now closed!
for name in files_to_create:
    path = tmpdir / name
    print(f"  📄 {name}: {path.read_text()}")

### ✏️ Exercise 11 — Retry with Exponential Backoff ⏳

In [ ]:
import time, random, logging

logger = logging.getLogger("Retry")

def retry_with_backoff(func, max_retries=5, base_delay=0.1):
    """Retry a function with exponential backoff."""
    for attempt in range(1, max_retries + 1):
        try:
            result = func()
            logger.info(f"✅ Success on attempt {attempt}")
            return result
        except Exception as e:
            delay = base_delay * (2 ** (attempt - 1))  # exponential!
            delay += random.uniform(0, delay * 0.1)    # jitter
            logger.warning(f"⚠️ Attempt {attempt}/{max_retries} failed: {e}. Retrying in {delay:.2f}s")
            if attempt == max_retries:
                logger.error(f"❌ All {max_retries} attempts failed")
                raise
            time.sleep(delay)

# Simulate flaky service
call_count = [0]
def flaky_service():
    call_count[0] += 1
    if call_count[0] < 3:
        raise ConnectionError("Server busy")
    return {"data": "success"}

result = retry_with_backoff(flaky_service)
print(f"Result: {result}")

### ✏️ Exercise 12 — EAFP vs LBYL Patterns 🎯

In [ ]:
# LBYL = Look Before You Leap (check first)
# EAFP = Easier to Ask Forgiveness (try/except) — PYTHONIC!

data = {"users": {"anita": {"age": 28, "scores": [85, 92, 78]}}}

# LBYL — verbose and fragile
if "users" in data:
    if "anita" in data["users"]:
        if "scores" in data["users"]["anita"]:
            if len(data["users"]["anita"]["scores"]) > 0:
                print(f"LBYL: {data['users']['anita']['scores'][0]}")

# EAFP — clean and Pythonic
try:
    print(f"EAFP: {data['users']['anita']['scores'][0]}")
except (KeyError, IndexError, TypeError) as e:
    print(f"Not found: {e}")

### ✏️ Exercise 13 — Custom Context Manager: Indent Printer 📐

In [ ]:
from contextlib import contextmanager

class IndentPrinter:
    """Context manager that adds indentation to print output."""
    _level = 0
    
    def __enter__(self):
        IndentPrinter._level += 1
        return self
    
    def __exit__(self, *args):
        IndentPrinter._level -= 1
        return False
    
    @staticmethod
    def print(msg):
        indent = "  " * IndentPrinter._level
        print(f"{indent}{msg}")

ip = IndentPrinter
ip.print("Level 0")
with IndentPrinter():
    ip.print("Level 1")
    with IndentPrinter():
        ip.print("Level 2")
        with IndentPrinter():
            ip.print("Level 3")
        ip.print("Back to 2")
    ip.print("Back to 1")
ip.print("Back to 0")

### ✏️ Exercise 14 — Exception Groups (Python 3.11+) 🎪

In [ ]:
# ExceptionGroup — handle multiple errors at once
import sys

if sys.version_info >= (3, 11):
    errors = [
        ValueError("bad value"),
        TypeError("wrong type"),
        KeyError("missing key"),
    ]
    
    try:
        raise ExceptionGroup("Multiple errors", errors)
    except* ValueError as eg:
        print(f"  ValueError: {eg.exceptions}")
    except* TypeError as eg:
        print(f"  TypeError: {eg.exceptions}")
    except* KeyError as eg:
        print(f"  KeyError: {eg.exceptions}")
else:
    print(f"  ℹ️ Python {sys.version_info.major}.{sys.version_info.minor} — ExceptionGroup needs 3.11+")
    print("  Simulating with a list of errors:")
    errors = [ValueError("bad"), TypeError("wrong")]
    for e in errors:
        print(f"    {type(e).__name__}: {e}")

---
## 🎯 Exercise 15 — Mini-Project: Resilient Data Processor 🏭

In [ ]:
from pathlib import Path
from contextlib import contextmanager
import logging, json, tempfile

# Setup logging
logger = logging.getLogger("DataProcessor")

# Custom exceptions
class ProcessingError(Exception): pass
class InvalidDataError(ProcessingError): pass

@contextmanager
def processing_pipeline(name):
    """Context manager for a data processing pipeline."""
    logger.info(f"🚀 Starting pipeline: {name}")
    stats = {"processed": 0, "errors": 0, "skipped": 0}
    try:
        yield stats
    except Exception as e:
        logger.error(f"💥 Pipeline '{name}' failed: {e}")
        raise
    finally:
        logger.info(f"📊 Pipeline '{name}' stats: {stats}")

def validate_record(record):
    if "name" not in record:
        raise InvalidDataError("Missing 'name' field")
    if not isinstance(record.get("age", 0), int):
        raise InvalidDataError(f"Age must be int, got {type(record.get('age')).__name__}")
    if record.get("age", 0) < 0:
        raise InvalidDataError(f"Age cannot be negative: {record['age']}")

def process_record(record, stats):
    try:
        validate_record(record)
        record["processed"] = True
        stats["processed"] += 1
        return record
    except InvalidDataError as e:
        stats["errors"] += 1
        logger.warning(f"  ⚠️ Skipping invalid record: {e}")
        return None

# Sample data
records = [
    {"name": "Anita", "age": 28},
    {"name": "Rahul", "age": -5},       # invalid!
    {"age": 30},                          # missing name!
    {"name": "Sneha", "age": "twenty"},  # wrong type!
    {"name": "Vishal", "age": 32},
    {"name": "Priya", "age": 25},
]

with processing_pipeline("UserImport") as stats:
    results = []
    for record in records:
        result = process_record(record, stats)
        if result:
            results.append(result)
    
    print(f"\n✅ Successfully processed {len(results)} records:")
    for r in results:
        print(f"   {r['name']} (age {r['age']})")

---
## 🎊 Session 6 Complete!

| ✅ | Concept | Key Takeaway |
|---|---------|-------------|
| 🏗️ | Custom exceptions | Hierarchy with `class MyError(Exception)` |
| 🎯 | try/except/else/finally | else=no error, finally=always |
| 🔗 | Exception chaining | `raise X from Y` |
| 🚪 | Context managers | `__enter__`/`__exit__`, `@contextmanager` |
| 📂 | pathlib | Modern file paths, `Path.read_text()`, `glob()` |
| 📋 | logging | Professional output with levels and handlers |
| 🤫 | suppress | Silently ignore specific exceptions |
| 📚 | ExitStack | Manage dynamic number of contexts |

### 🚀 Next: Session 7 — Testing, Type Hints & Capstone
---

## 📖 Solutions

<details>
<summary><strong>Click to reveal all solutions</strong></summary>

### Ex 2: `42` → Parsing → ✅ Got 42 → 🔚 Done → 42. `abc` → Parsing → ❌ Not a number → 🔚 Done → None.
### Ex 5: Normal: connect → execute → commit → disconnect. Error: connect → execute → rollback → disconnect.
### Ex 12: EAFP (try/except) is the Pythonic way.

</details>